In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
import pickle
import deepRD.tools.trajectoryTools as trajectoryTools
import csv
import math
from deepRD.noiseSampler import noiseSampler
from torchvision.transforms import ToTensor
from torch import nn
from torch.utils.data import DataLoader
from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
from scipy.stats import gaussian_kde, wasserstein_distance_nd

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
# System type: 'bistable', 'dimer'
systemType = 'dimerGlobal'

# datapoint = [time (1), qi (3), vi (3), ? (1), ri(3)] -- 11 dim for benchmark
# for dimer, alternating between particle 1 and particle 2.

#for generated trajs, 
# datapoint = [time (1), qi (3), vi (3), ri(3)] - 10 dim

# Datasets directory
localDirectoryBase = "/group/ag_cmb/scratch/maojrs/stochasticClosure/" + systemType + "/boxsize5/"

# loading multiple simulation folders for comparisons, and corresponding labels
# Conditioning variables: piri, piririm, pipimri, etc. - for dimer, piridqi, 'pipimdqiririm' #'dqidpiririm'
datasetFolders = ["benchmarkReducedGen_pipimririm/", "benchmarkReducedGen2_pipimririm/"]#,#["benchmarkReducedGen_pipimdpiririm/", "benchmarkReducedGen_pipimdqidpiririm/", "benchmarkReducedGen_pipimririm/"]#, "benchmarkReducedGen_pidqiri"]
datasetLabels = ['pipimririm', 'pipimririm, Tz=0'] #['pipimdpiririm', 'pipimdqidpiririm', 'pipimririm'] #, "pidqiri"]
nModels = len(datasetFolders)

nTrajs = 100 # no. of trajectories to load per data folder

nTimestepsBench = 10000 # length of benchmark trajectory
nTimesteps = 200000 # length of coarse-grained trajectory
datapointDim = 10 # dimensionality of datapoints.
numFilesBench = 2500
numFiles = 100 # total no. of files in data folder

# Initialising tensor to store trajectories
dataset = torch.empty((nModels, nTrajs, 2*nTimesteps, datapointDim))
    
# Loading reduced models data

truncate = False # if True, will select random fragment of trajectory

for i, datasetFolder in enumerate(datasetFolders):
    
    filePath = localDirectoryBase + datasetFolder + "simMoriZwanzigReduced_"    
    
    fileIDS = np.sort(np.random.choice(numFiles, nTrajs, replace=False))
    if i==1:
        print(fileIDS)
    
    for j, fnum in enumerate(fileIDS):
        try:
            ds = torch.Tensor(trajectoryTools.loadTrajectory(filePath, fnum))
        except FileNotFoundError:
            print(f'{datasetFolder}: File {fnum} not available', end='\r')
            continue
            
        if ds.shape[1]==7:
            ds = torch.cat((ds, torch.zeros(2*nTimesteps,3)), dim=-1)
        dataset[i, j] = ds
        print(f'Model {i+1}: File no. {fnum} loaded', end='\r')

#dataset_norm = (dataset[0] - mean_d)/std_d
dataset.shape # nModels, nTrajs, nTimesteps, datapointDims

In [ ]:
# Loading Benchmark data

benchDataset = torch.empty((nTrajs, 2*nTimestepsBench, datapointDim))

fileIDS = np.sort(np.random.choice(numFilesBench, nTrajs, replace=False))
benchLocalDirectoryBase = "/group/ag_cmb/scratch/maojrs/stochasticClosure/dimer/boxsize5/"

benchFileDirectory = benchLocalDirectoryBase + "benchmark/simMoriZwanzig_"

for j, fnum in enumerate(fileIDS):
    try:
        ds = torch.Tensor(trajectoryTools.loadTrajectory(benchFileDirectory, fnum))
    except FileNotFoundError:
        print(f'File {fnum} not available')
        continue
    print(f'File no. {fnum} loaded', end='\r')
    # cutting out meaningless variable
    if ds.shape[1]==11:
        ds = torch.cat((ds[:, :7], ds[:, -3:]), dim=1)
        #ds = ds[:, :7]
    
    benchDataset[j] = ds
    
benchDataset.shape

In [ ]:
b_timesteps = benchDataset[:, ::2, 0]
b_qT = torch.cat((benchDataset[:, ::2, 1:4], benchDataset[:, 1::2, 1:4]), dim=-1)
b_rAuxT = torch.cat((benchDataset[:, ::2, -3:], benchDataset[:, 1::2, -3:]), dim=-1)
b_rNxtT = torch.roll(b_rAuxT, -1, 1)
b_vT = torch.cat((benchDataset[:, ::2, 4:7], benchDataset[:, 1::2, 4:7]), dim=-1)

# 0:3 - particle 1, 3:6 - particle 2
timesteps = dataset[:, :, ::2, 0]
qT = torch.cat((dataset[:, :, ::2, 1:4], dataset[:, :, 1::2, 1:4]), dim=-1)
rAuxT = torch.cat((dataset[:, :, ::2, -3:], dataset[:, :, 1::2, -3:]), dim=-1)
rNxtT = torch.roll(rAuxT, -1, 1)
vT = torch.cat((dataset[:, :, ::2, 4:7], dataset[:, :, 1::2, 4:7]), dim=-1)

b_timesteps.shape, b_qT.shape, b_rAuxT.shape, b_rNxtT.shape, b_vT.shape, timesteps.shape, qT.shape, rAuxT.shape, rNxtT.shape, vT.shape

In [ ]:
print('\nVelocity')

print(f'Mean Bench: \t', torch.mean(b_vT, dim=(0,1)))
for i in range(nModels):
    print(f'Mean Reduced {i+1}:\t', torch.mean(vT[i], dim=(0,1)))

print('\nStd Bench: \t', torch.std(b_vT, dim=(0,1)))
for i in range(nModels):
    print(f'Std Reduced {i+1}:\t', torch.std(vT[i], dim=(0,1)))
    
print('\n Auxiliary Var')
print('Mean Bench: \t', torch.mean(b_rAuxT, dim=(0,1)))
for i in range(nModels):
    print(f'Mean Reduced {i+1}:\t', torch.mean(rAuxT[i], dim=(0,1)))

print('\nStd Bench: \t', torch.std(b_rAuxT, dim=(0,1)))
for i in range(nModels):
    print(f'Std Reduced {i+1}:\t', torch.std(rAuxT[i], dim=(0,1)))

In [ ]:
def compute_delta_x_loop(q1, q2, boxsize=5.0, boundary_type="periodic"):
    """
    Compute Δx = ||relativePosition(q1, q2)|| using the original loop-based
    trajectoryTools.relativePosition function.
    
    Returns
    -------
    delta_x : np.ndarray of shape [n_traj, T]
    """
    
    # convert to numpy arrays
    q1_np = q1.cpu().numpy()
    q2_np = q2.cpu().numpy()

    n_traj, T, _ = q1_np.shape
    delta_x = np.zeros((n_traj, T), dtype=np.float64)

    for i in range(n_traj):
        for j in range(T):
            relPos = trajectoryTools.relativePosition(
                q1_np[i, j],
                q2_np[i, j],
                boundary_type,
                boxsize
            )
            delta_x[i, j] = np.linalg.norm(relPos)

    return delta_x


def minimal_image_rel(q1, q2, boxsize=None, boundary_type='periodic'):
    """
    q1, q2: [..., 3] torch tensors
    returns q2 - q1 with minimal-image convention matching trajectoryTools.relativePosition
    """
    rel = q2 - q1  # [..., 3]

    if boundary_type == "periodic" and boxsize is not None:
        # box: tensor of shape [3]
        if isinstance(boxsize, (list, tuple, np.ndarray)):
            box = torch.tensor(boxsize, dtype=rel.dtype, device=rel.device)
        else:  # scalar -> same in all dims
            box = torch.full((3,), float(boxsize), dtype=rel.dtype, device=rel.device)

        # broadcast box over leading dims, minimal image per component
        rel = rel - box * torch.round(rel / box)

    return rel

def compute_axisRelVel(q1, q2, v1, v2, boxsize, boundary_type="periodic"):
    """
    Computes axis-relative velocity for a dimer in a fully vectorised way.

    Inputs:
      q1, q2: positions  [..., 3]
      v1, v2: velocities [..., 3]
      boxsize: scalar or 3-vector
      boundary_type: "periodic" or "none"

    Returns:
      axisRelVel: tensor [..., 1]
    """

    # Minimal-image relative position
    rel_pos = minimal_image_rel(q1, q2, boxsize, boundary_type)   # [..., 3]

    # Relative velocity
    rel_vel = v2 - v1                                             # [..., 3]

    # Unit vector along dimer axis
    norm_rel = torch.norm(rel_pos, dim=-1, keepdim=True)          # [..., 1]
    unit_rel = rel_pos / (norm_rel + 1e-12)                       # avoid div-by-zero

    # Projection of relative velocity onto dimer axis
    axis_rel_vel = torch.sum(rel_vel * unit_rel, dim=-1, keepdim=False)  # [..., 1]

    return axis_rel_vel

def plot_dimer_distributions(delta_x, delta_vx, cdx=None, cdv=None):
    """
    Plot KDEs of Δx (left) and axis-relative velocity Δv_x (right),
    optionally overlaying comparison distributions.

    Args:
        delta_x:     array-like, Δx samples (any shape; will be flattened)
        delta_vx:    array-like, axisRelVel samples (any shape; will be flattened)
        comp_dx:     optional array-like, comparison Δx samples
        comp_dvx:    optional array-like, comparison axisRelVel samples
    """

    # KDEs
    kde_dx  = gaussian_kde(delta_x)
    kde_dvx = gaussian_kde(delta_vx)

    # grids
    xmin = dx.min() if cdx is None else min(dx.min(), cdx.min())
    xmax = dx.max() if cdx is None else max(dx.max(), cdx.max())
    xs_dx = np.linspace(xmin, xmax, 100)

    vmin = dvx.min() if cdv is None else min(dvx.min(), cdv.min())
    vmax = dvx.max() if cdv is None else max(dvx.max(), cdv.max())
    xs_dv = np.linspace(vmin, vmax, 100)

    # plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # ---- LEFT: Δx ----
    axes[0].plot(xs_dx, kde_dx(xs_dx), lw=2, label="Δx")
    if cdx is not None:
        kde_cdx = gaussian_kde(cdx)
        axes[0].plot(xs_dx, kde_cdx(xs_dx), lw=2, linestyle="--", label="Δx (comparison)")
    axes[0].set_title("Δx distribution")
    axes[0].set_xlabel("Δx")
    axes[0].set_ylabel("Density")
    axes[0].grid(alpha=0.2)
    axes[0].legend()

    # ---- RIGHT: axisRelVel ----
    axes[1].plot(xs_dv, kde_dvx(xs_dv), lw=2, label="axisRelVel")
    if cdv is not None:
        kde_cdv = gaussian_kde(cdv)
        axes[1].plot(xs_dv, kde_cdv(xs_dv), lw=2, linestyle="--", label="axisRelVel (comparison)")
    axes[1].set_title("Axis-relative velocity distribution")
    axes[1].set_xlabel("axisRelVel")
    axes[1].set_ylabel("Density")
    axes[1].grid(alpha=0.2)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_dimer_distributions(delta_x, delta_vx, cdx=None, cdv=None):
    """
    Plot KDEs of Δx (left) and axis-relative velocity Δv_x (right),
    with optional multiple comparison models.

    Args:
        delta_x : array-like, benchmark Δx samples (any shape; flattened internally)
        delta_vx: array-like, benchmark axisRelVel samples (any shape; flattened internally)
        cdx     : optional array-like, shape (nModels, nTrajs, nTimesteps) for Δx
        cdv     : optional array-like, shape (nModels, nTrajs, nTimesteps) for axisRelVel
    """

    def _to_flat_np(x):
        if x is None:
            return None
        if hasattr(x, "detach"):  # torch.Tensor
            x = x.detach().cpu().numpy()
        x = np.asarray(x)
        return x.reshape(-1)

    # --- benchmark, flattened ---
    dx_bench  = _to_flat_np(delta_x)
    dvx_bench = _to_flat_np(delta_vx)

    # --- models: list of 1D arrays per model ---
    cdx_models = []
    if cdx is not None:
        cdx_np = cdx.detach().cpu().numpy() if hasattr(cdx, "detach") else np.asarray(cdx)
        for i in range(cdx_np.shape[0]):
            cdx_models.append(cdx_np[i].reshape(-1))

    cdv_models = []
    if cdv is not None:
        cdv_np = cdv.detach().cpu().numpy() if hasattr(cdv, "detach") else np.asarray(cdv)
        for i in range(cdv_np.shape[0]):
            cdv_models.append(cdv_np[i].reshape(-1))

    # --- KDEs for benchmark ---
    kde_dx_bench  = gaussian_kde(dx_bench)
    kde_dvx_bench = gaussian_kde(dvx_bench)

    # --- grids ---
    # Δx grid bounds
    xmin = dx_bench.min()
    xmax = dx_bench.max()
    for arr in cdx_models:
        xmin = min(xmin, arr.min())
        xmax = max(xmax, arr.max())
    xs_dx = np.linspace(xmin, xmax, 100)

    # Δv grid bounds
    vmin = dvx_bench.min()
    vmax = dvx_bench.max()
    for arr in cdv_models:
        vmin = min(vmin, arr.min())
        vmax = max(vmax, arr.max())
    xs_dv = np.linspace(vmin, vmax, 100)

    # --- plot ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # ---- LEFT: Δx ----
    axes[0].plot(xs_dx, kde_dx_bench(xs_dx), lw=2, label="Δx (benchmark)")
    for i, arr in enumerate(cdx_models):
        kde_model = gaussian_kde(arr)
        axes[0].plot(xs_dx, kde_model(xs_dx), lw=1.5, linestyle="--", label=f"Δx ({datasetLabels[i]})")

    axes[0].set_title("Δx distribution")
    axes[0].set_xlabel("Δx")
    axes[0].set_ylabel("Density")
    axes[0].grid(alpha=0.2)
    axes[0].legend()

    # ---- RIGHT: axis-relative velocity ----
    axes[1].plot(xs_dv, kde_dvx_bench(xs_dv), lw=2, label="axisRelVel (benchmark)")
    for i, arr in enumerate(cdv_models):
        kde_model = gaussian_kde(arr)
        axes[1].plot(xs_dv, kde_model(xs_dv), lw=1.5, linestyle="--", label=f"axisRelVel ({datasetLabels[i]})")

    axes[1].set_title("Axis-relative velocity distribution")
    axes[1].set_xlabel("axisRelVel")
    axes[1].set_ylabel("Density")
    axes[1].grid(alpha=0.2)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def compute_dx_dvx(qT, vT, flatten=True):
    # Reduced model

    # split particles
    q1, q2 = qT[..., :3], qT[..., 3:]
    v1, v2 = vT[..., :3], vT[..., 3:]

    # Δx 
    rel = minimal_image_rel(q1, q2, boundary_type='periodic', boxsize=5.0)
    delta_x = torch.norm(rel, dim=-1)                       # [n_traj, T]

    # axis-relative velocity
    delta_vx = compute_axisRelVel(q1, q2, v1, v2, boxsize=5.0)
    
    if flatten:
        delta_x = delta_x.reshape(-1).cpu().numpy()
        delta_vx = delta_vx.reshape(-1).cpu().numpy()
        
    return delta_x, delta_vx

In [ ]:
# split particles
b_q1, b_q2 = b_qT[..., :3], b_qT[..., 3:]
b_v1, b_v2 = b_vT[..., :3], b_vT[..., 3:]

compare=False
    
# --------------------------------------
# Δx vectorised
# --------------------------------------
rel = minimal_image_rel(b_q1, b_q2, boundary_type='periodic', boxsize=5.0)
b_delta_x = torch.norm(rel, dim=-1)                       # [n_traj, T]
b_dx = b_delta_x.reshape(-1).cpu().numpy()

In [ ]:
# --------------------------------------
# axis-relative velocity
# --------------------------------------
b_delta_vx = compute_axisRelVel(b_q1, b_q2, b_v1, b_v2, boxsize=5.0)
b_dvx = b_delta_vx.reshape(-1).cpu().numpy()

In [ ]:
# Reduced model

# split particles
q1, q2 = qT[..., :3], qT[..., 3:]
v1, v2 = vT[..., :3], vT[..., 3:]

# Δx 
rel = minimal_image_rel(q1, q2, boundary_type='periodic', boxsize=5.0)
delta_x = torch.norm(rel, dim=-1)                       # [n_traj, T]

# axis-relative velocity
delta_vx = compute_axisRelVel(q1, q2, v1, v2, boxsize=5.0)

In [ ]:
b_delta_x.shape, b_delta_vx.shape, delta_x.shape, delta_vx.shape

In [ ]:
print('Relative Position')

print(f'Bench: \t\t Mean:', torch.mean(b_delta_x), '\t Std:', torch.std(b_delta_x))
for i in range(nModels):
    print(f'Reduced {i+1}: \t Mean:', torch.mean(delta_x[i]), '\t Std:', torch.std(delta_x[i]))

print('\nRelative Velocity')

print(f'Bench: \t\t Mean:', torch.mean(b_delta_vx), '\t Std:', torch.std(b_delta_vx))
for i in range(nModels):
    print(f'Reduced {i+1}: \t Mean:', torch.mean(delta_vx[i]), '\t Std:', torch.std(delta_vx[i]))

In [ ]:
plot_dimer_distributions(b_delta_x, b_delta_vx, delta_x, delta_vx)

In [ ]:
def correlation_fft(a, b, trunc):
    """Calculates correlation via FFT."""
    len_a = len(a)
    a = np.concatenate([a, np.zeros(len_a)])
    b = np.concatenate([b, np.zeros(len_a)])
    a_fft = np.fft.fft(a)
    b_fft = np.fft.fft(b)
    corr = np.fft.ifft(a_fft * np.conj(b_fft))
    corr = corr[:trunc].real
    corr /= np.linspace(len_a, len_a - trunc + 1, trunc)
    return corr

In [ ]:
lagtimesteps = 2000
mTrajs = 20

# ACF by FFT for 1-D traj, then summing up over vector dimensions and all trajs 
ACF_FFT = np.zeros((1+nModels, 2, lagtimesteps))
    
for trajInd in np.random.choice(nTrajs, mTrajs):
    
    #print('Benchmark')
    # position
    ACF_FFT[0, 0] += correlation_fft(b_delta_x[trajInd, :], b_delta_x[trajInd, :], lagtimesteps)
    # velocity
    ACF_FFT[0, 1] += correlation_fft(b_delta_vx[trajInd, :], b_delta_vx[trajInd, :], lagtimesteps)
    
    #print('Reduced')
    for i in range(nModels):
        # position
        ACF_FFT[i+1, 0] += correlation_fft(delta_x[i, trajInd, :], delta_x[i, trajInd, :], lagtimesteps)
        # velocity
        ACF_FFT[i+1, 1] += correlation_fft(delta_vx[i, trajInd, :], delta_vx[i, trajInd, :], lagtimesteps)
    
for j in range(ACF_FFT.shape[0]):
    ACF_FFT[j, 0] /= ACF_FFT[j, 0, 0]
    ACF_FFT[j, 1] /= ACF_FFT[j, 1, 0]

In [ ]:
ACF_FFT.shape

In [ ]:
# lag axis
lags = np.arange(lagtimesteps)*0.05

# ACF arrays:
# ACF_FFT[0, 0] → Δx ACF
# ACF_FFT[0, 1] → Δv ACF
acf_dx = ACF_FFT[:, 0]
acf_dvx = ACF_FFT[:, 1]

# ----- PLOTS -----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Left: relative distance ACF ---
axes[0].plot(lags, acf_dx[0], lw=2, label='benchmark')#, color="darkblue")
axes[0].set_title("Relative Distance Autocorrelation")
axes[0].set_xlabel("Lag [ns]")
axes[0].set_ylabel("ACF")
axes[0].grid(alpha=0.3)

# --- Right: relative velocity ACF ---
axes[1].plot(lags, acf_dvx[0], lw=2, label='benchmark')#, color="darkred")
axes[1].set_title("Relative Velocity Autocorrelation")
axes[1].set_xlabel("Lag [ns]")
axes[1].set_ylabel("ACF")
axes[1].grid(alpha=0.3)

for j in range(1, ACF_FFT.shape[0]):
    axes[0].plot(lags, acf_dx[j], '--', lw=2, label=datasetLabels[j-1])
    axes[1].plot(lags, acf_dvx[j], '--', lw=2, label=datasetLabels[j-1])
    
axes[0].legend()
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#Calculating the Mean Squared Displacement of the particle.
b_rT = torch.norm(b_qT, dim=-1)
b_diffsq = torch.diff(b_rT, dim=-1)**2
b_MSD = torch.mean(b_diffsq)

rT = torch.norm(qT, dim=-1)
diffsq = torch.diff(rT, dim=-1)**2
MSD = torch.mean(diffsq, dim=(1,2))

print('Mean Square Displacement:')
print('Benchmark:', float(b_MSD))

for i in range(MSD.shape[0]):
    print(f'Model {i+1}:', float(MSD[i]))

In [ ]:
def bondframe_components(q: torch.Tensor, r: torch.Tensor, eps: float = 1e-12):
    """
    Compute r_parallel and r_perp magnitudes in the bond-aligned frame.
    Works for both bench [T,B,...] and reduced [M,T,B,...] as long as last dim=6.
    Returns:
        rpar: [..., 2]  (per monomer)
        rperp: [..., 2] (per monomer)
    """
    # split particles: [..., 3]
    q1, q2 = q[..., :3], q[..., 3:]
    r1, r2 = r[..., :3], r[..., 3:]

    # bond axis (no PBC here; if you need PBC, replace rel_pos with minimal_image_rel(q1,q2,...))
    rel_pos = minimal_image_rel(q1, q2, boundary_type="periodic", boxsize=5.0)
    rel_norm = torch.linalg.norm(rel_pos, dim=-1, keepdim=True).clamp_min(eps)
    u = rel_pos / rel_norm                             # [..., 3]

    # parallel components (scalar)
    r1_par = torch.sum(r1 * u, dim=-1, keepdim=True)   # [..., 1]
    r2_par = torch.sum(r2 * u, dim=-1, keepdim=True)   # [..., 1]

    # perpendicular magnitudes
    r1_perp = torch.linalg.norm(r1 - r1_par * u, dim=-1, keepdim=True)  # [..., 1]
    r2_perp = torch.linalg.norm(r2 - r2_par * u, dim=-1, keepdim=True)  # [..., 1]

    # pack per-monomer: [..., 2]
    rpar  = torch.cat([r1_par,  r2_par],  dim=-1)
    rperp = torch.cat([r1_perp, r2_perp], dim=-1)
    return rpar, rperp


In [ ]:
# ---- benchmark ----
rpar_b, rperp_b = bondframe_components(b_qT, b_rAuxT)   # [nTrajs, nTimesteps, 2]

# ---- reduced models ----
rpar_m, rperp_m = bondframe_components(qT, rAuxT)       # [nModels, nTrajs, nTimestepsReduced, 2]

# ---- quick summary stats (useful sanity check) ----
print("bench r_parallel mean/std:", rpar_b.mean().item(), rpar_b.std().item())
print("models r_parallel mean/std:", rpar_m.mean(dim=(1,2,3)).cpu(), rpar_m.std(dim=(1,2,3)).cpu())
print("bench r_perp     mean/std:", rperp_b.mean().item(), rperp_b.std().item())
print("models r_perp     mean/std:", rperp_m.mean(dim=(1,2,3)).cpu(), rperp_m.std(dim=(1,2,3)).cpu())

# If you want 1D flattened arrays for KDE later:
rpar_b_flat  = rpar_b.reshape(-1)          # benchmark scalars (both monomers pooled)
rperp_b_flat = rperp_b.reshape(-1)

rpar_m_flat  = rpar_m.reshape(rpar_m.shape[0], -1)    # [nModels, N]
rperp_m_flat = rperp_m.reshape(rperp_m.shape[0], -1)  # [nModels, N]